# 07 · External Validation — Hansen Global Forest Change

**Decision D-015.** Cross-validation (D-014) only proves the model agrees with *its own labels*.
This notebook checks whether our results agree with an **independent, peer-reviewed** dataset:
the **Hansen Global Forest Change** map (Hansen et al., *Science* 2013; University of Maryland +
Google, hosted in Earth Engine — annual tree-cover loss at 30 m).

**What we do:** for every Rwandan sector, take Hansen's measured forest loss (2020–2023) and line
it up against our `deforested_pct`. If the sectors *we* flag match the ones *Hansen* flags, that is
strong external evidence the results are real — not an artefact of our labelling.

**Outputs (for the dissertation):**
- `results/metrics/hansen_scatter.png` — our values vs Hansen, with correlation
- `results/metrics/hansen_vs_ours_maps.png` — side-by-side choropleths
- `results/metrics/hansen_loss_map.png` — the actual Hansen loss image over Rwanda
- `results/metrics/hansen_benchmark.json` — the numbers (r, rho, per-sector table)

> **Limitation to state in the report:** Hansen v1.11 covers loss only through **2023**; our
> training window is 2020–2024, so the final year is not yet in the benchmark.

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ee

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RISK_JSON   = ROOT / 'results' / 'application' / 'sector_risk.json'
SECTORS_GEO = ROOT / 'data' / 'geo' / 'sectors_wgs84.geojson'
FIG_DIR     = ROOT / 'results' / 'metrics'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Earth Engine init ──
# If ee.Initialize() complains about a project, set EE_PROJECT to your Google
# Cloud project id (free for research). Otherwise leave it as None.
EE_PROJECT = None
try:
    ee.Initialize(project=EE_PROJECT) if EE_PROJECT else ee.Initialize()
    print('Earth Engine ready')
except Exception as e:
    print('EE NOT authenticated. In a terminal run:  earthengine authenticate')
    print('then re-run this cell. If it asks for a project, set EE_PROJECT above.')
    raise

EE NOT authenticated. In a terminal run:  earthengine authenticate
then re-run this cell. If it asks for a project, set EE_PROJECT above.


EEException: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.

## 1 · Load our sector results

In [ ]:
risk = json.loads(RISK_JSON.read_text())
ours = pd.DataFrame(risk['sectors'])
ours['sector_id'] = ours['sector_id'].astype(str)
print(f"{len(ours)} assessed sectors  |  model: {risk['model_version']}  |  generated: {risk['generated_at']}")
ours[['sector_id', 'sector', 'district', 'deforested_pct', 'n_pixels', 'risk_level']].head()

## 2 · Query Hansen forest loss per sector (Earth Engine)

For each sector polygon we compute the **fraction of its area that Hansen recorded as forest loss
in 2020–2023** (`lossyear` codes 20–23). The mean of a 0/1 loss mask over a region = that fraction.

In [ ]:
gfc = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
lossyear = gfc.select('lossyear')
loss_in_window = lossyear.gte(20).And(lossyear.lte(23)).rename('hansen_frac')

# Our sector polygons -> ee.FeatureCollection (carry sector_id through)
geo = json.loads(SECTORS_GEO.read_text())
feats = [
    ee.Feature(ee.Geometry(f['geometry']), {'sector_id': str(f['properties']['sector_id'])})
    for f in geo['features']
]
fc = ee.FeatureCollection(feats)

# Server-side mean loss fraction per sector at Hansen's native 30 m
reduced = loss_in_window.reduceRegions(
    collection=fc, reducer=ee.Reducer.mean(), scale=30, tileScale=4,
)

rows = []
for f in reduced.getInfo()['features']:
    p = f['properties']
    frac = p.get('mean')
    if frac is not None:
        rows.append({'sector_id': str(p['sector_id']), 'hansen_pct': round(float(frac) * 100, 4)})
hansen = pd.DataFrame(rows)
print(f'Hansen returned loss for {len(hansen)} sectors')
hansen.head()

## 3 · Agreement — the headline number

In [ ]:
df = ours.merge(hansen, on='sector_id', how='inner')
df = df[df['n_pixels'] >= 5].copy()   # only reliably-sampled sectors

pear  = df['deforested_pct'].corr(df['hansen_pct'], method='pearson')
spear = df['deforested_pct'].corr(df['hansen_pct'], method='spearman')

print(f'Sectors compared : {len(df)}')
print(f'Pearson  r       : {pear:.3f}')
print(f'Spearman rho     : {spear:.3f}')
print()
print('Interpretation: r > 0.5 = moderate, > 0.7 = strong agreement with the independent benchmark.')

## 4 · Figure 1 — scatter: ours vs Hansen

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df['hansen_pct'], df['deforested_pct'], s=20, alpha=0.5, edgecolor='none')
m, b = np.polyfit(df['hansen_pct'], df['deforested_pct'], 1)
xs = np.linspace(df['hansen_pct'].min(), df['hansen_pct'].max(), 50)
ax.plot(xs, m * xs + b, color='crimson', lw=2, label=f'fit  (Pearson r = {pear:.2f})')
ax.set_xlabel('Hansen forest loss 2020–2023  (% of sector area)')
ax.set_ylabel('Our deforested_pct  (% of sampled pixels)')
ax.set_title('External validation: our results vs Hansen GFC')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'hansen_scatter.png', dpi=150)
plt.show()

## 5 · Figure 2 — side-by-side maps (ours vs Hansen)

In [ ]:
sectors = gpd.read_file(SECTORS_GEO)
sectors['sector_id'] = sectors['sector_id'].astype(str)
g = sectors.merge(df[['sector_id', 'deforested_pct', 'hansen_pct']], on='sector_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
g.plot(column='deforested_pct', cmap='OrRd', legend=True, ax=axes[0],
       missing_kwds={'color': 'lightgrey', 'label': 'not assessed'})
axes[0].set_title('Ours — deforested_pct'); axes[0].axis('off')
g.plot(column='hansen_pct', cmap='OrRd', legend=True, ax=axes[1],
       missing_kwds={'color': 'lightgrey', 'label': 'no data'})
axes[1].set_title('Hansen — forest loss 2020–2023'); axes[1].axis('off')
fig.tight_layout()
fig.savefig(FIG_DIR / 'hansen_vs_ours_maps.png', dpi=150)
plt.show()

## 6 · Figure 3 — the actual Hansen loss image over Rwanda

This is the independent satellite-derived map itself (forest in dark green, 2020–2023 loss in red)
— useful as a visual in the report alongside our risk map.

In [ ]:
import urllib.request
from IPython.display import Image as IPImage

region = fc.geometry().bounds()
bg       = gfc.select('treecover2000').visualize(min=0, max=100, palette=['000000', '1a3d1a', '2e7d32'])
loss_vis = loss_in_window.selfMask().visualize(palette=['ff2d2d'])
blend    = bg.blend(loss_vis)

url = blend.getThumbURL({'region': region, 'dimensions': 1024, 'format': 'png'})
out_png = FIG_DIR / 'hansen_loss_map.png'
urllib.request.urlretrieve(url, out_png)
print('saved ->', out_png)
IPImage(filename=str(out_png))

## 7 · Where we disagree most (also a finding)

Large gaps are worth discussing: e.g. Hansen often **misses small farm clearings** that our model
catches — which actually supports the motivation for this project.

In [ ]:
df['abs_diff'] = (df['deforested_pct'] - df['hansen_pct']).abs()
top = (df.sort_values('abs_diff', ascending=False)
         [['sector', 'district', 'risk_level', 'deforested_pct', 'hansen_pct', 'n_pixels']]
         .head(12).round(1))
display(top)

out = {
    'benchmark':          'Hansen Global Forest Change (UMD/Google, Science 2013)',
    'hansen_asset':       'UMD/hansen/global_forest_change_2023_v1_11',
    'loss_window':        '2020-2023 (Hansen v1.11 max; training window is 2020-2024)',
    'n_sectors_compared': int(len(df)),
    'pearson_r':          float(pear),
    'spearman_rho':       float(spear),
    'per_sector':         df.sort_values('deforested_pct', ascending=False)[
                              ['sector_id', 'sector', 'district', 'risk_level',
                               'deforested_pct', 'hansen_pct', 'n_pixels']
                          ].round(3).to_dict(orient='records'),
}
(FIG_DIR / 'hansen_benchmark.json').write_text(json.dumps(out, indent=2))
print('saved ->', FIG_DIR / 'hansen_benchmark.json')

## 8 · What to write in the report

- **One sentence for the defence:** *"Per-sector deforestation from our system correlates with the
  independent Hansen Global Forest Change benchmark at r = … (n = … sectors), providing external
  validation beyond internal cross-validation."*
- Put **Figure 2** (side-by-side maps) and **Figure 3** (Hansen loss map) in the Results chapter.
- Discuss the **disagreements** honestly — small-clearing detection is where the methods diverge,
  and that is the gap this project targets.
- State the **limitation**: Hansen covers to 2023; our window runs to 2024.